# Experiment 001: 0.5B validation run (Colab)

Runs the full pipeline (base -> sft -> sft-then-grpo -> eval, plus the logprob
knowledge probe) on Qwen2.5-0.5B and logs everything.

**Note:** this is the stage 1 validation run: it proves the pipeline on a real gpu,
measures cost for the budget check, and its curves decide the protocol freeze.
The reportable 0.5b numbers come from a fresh rerun after the freeze.

Before starting: Runtime -> Change runtime type -> GPU (T4 works, L4 is faster).


## 1. Check the GPU


In [1]:
!nvidia-smi -L
import torch
print('cuda:', torch.cuda.is_available(), '| torch:', torch.__version__)


GPU 0: Tesla T4 (UUID: GPU-6af08d9e-ee77-d9d2-67e7-31a9e40f16f1)
cuda: True | torch: 2.11.0+cu128


## 2. Clone the repo (automatic)

This cell clones the main branch of the public repository without asking for a GitHub token. For online experiment tracking, add `WANDB_API_KEY` under the key icon in Colab Secrets and grant this notebook access. If using a private fork, set `REPO_SLUG` to its owner and name, then add a read-only `GH_TOKEN` for that fork. A missing token for a private fork triggers a one-time prompt.


In [2]:
import base64
import getpass
import glob
import os
from pathlib import Path
import subprocess

REPO_SLUG = 'hastov-lab/rl-grpo-or-sft-for-llms'
repo_url = f'https://github.com/{REPO_SLUG}.git'
repo_root = Path('/content') / REPO_SLUG.split('/')[-1]


def repo_git(*args):
    git = ['git']
    env = None
    try:
        from google.colab import userdata
    except ImportError:
        userdata = None

    # Private forks can use a read token from Colab Secrets.
    if userdata is not None and REPO_SLUG != 'hastov-lab/rl-grpo-or-sft-for-llms':
        try:
            token = userdata.get('GH_TOKEN')
        except Exception:
            token = None
        if isinstance(token, dict):
            token = token.get('value')
        if not isinstance(token, str) or not token.strip():
            token = getpass.getpass('GitHub token: ')
        if not token.strip():
            raise RuntimeError('This private fork requires a GitHub token with read access.')

        # Keep the token out of the command line and the saved remote URL.
        auth = base64.b64encode(f'x-access-token:{token.strip()}'.encode()).decode()
        env = dict(os.environ, REPO_GIT_AUTH=f'Authorization: Basic {auth}')
        git.append('--config-env=http.https://github.com/.extraHeader=REPO_GIT_AUTH')

    subprocess.run([*git, *args], env=env, check=True)


if repo_root.exists():
    if not (repo_root / '.git').exists():
        raise RuntimeError(f'Expected a Git checkout at {repo_root}.')
    origin = subprocess.check_output(
        ['git', '-C', str(repo_root), 'config', '--local', '--get', 'remote.origin.url'], text=True
    ).strip()
    if origin != repo_url:
        raise RuntimeError(f'The checkout at {repo_root} belongs to a different repository.')
else:
    repo_git('clone', '--branch', 'main', '--single-branch', '--depth', '1', repo_url, str(repo_root))

# Use the scripts from this checkout even if another experiment is open.
code_root = repo_root / '001_Code'
if not (code_root / 'scripts' / 'run.py').is_file():
    raise RuntimeError(f'Experiment scripts are missing from {code_root}.')
os.chdir(code_root)
print('working in:', os.getcwd())


Cloning into 'INM363-IndividualProject-Dissertation'...
remote: Enumerating objects: 389, done.
remote: Counting objects: 100% (389/389), done.
remote: Compressing objects: 100% (245/245), done.
remote: Total 389 (delta 215), reused 307 (delta 138), pack-reused 0 (from 0)
Receiving objects: 100% (389/389), 118.66 KiB | 1021.00 KiB/s, done.
Resolving deltas: 100% (215/215), done.
working in: /content/INM363-IndividualProject-Dissertation/001_Code


## 3. Install dependencies

Keep colabs own cuda torch, just pin the libraries that drifted.


In [3]:
!pip install -q trl==1.7.1 transformers==5.5.4 peft==0.19.1 accelerate==1.14.0 datasets==4.8.4 wandb pyyaml
# Colab preinstalls a torchao that peft 0.19.1 chokes on inside its lora
# dispatcher. No quantization here, so just drop it
!pip uninstall -q -y torchao
print('deps installed')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 104.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.6 MB/s eta 0:00:00
deps installed


## 4. W&B key (automatic)

Reads `WANDB_API_KEY` from Colab Secrets and writes it to the file the code expects.
If the secret isn't set, it falls back to a manual upload of `wandb_api_key.txt`.


In [4]:
import os

# 1) Use the key file when it is already present.
# 2) Else read WANDB_API_KEY from colab secrets and write the file.
# 3) Else fall back to a manual upload.
if os.path.exists('wandb_api_key.txt'):
    print('wandb_api_key.txt already present, using it')
else:
    wkey = None
    try:
        from google.colab import userdata
        wkey = userdata.get('WANDB_API_KEY')
    except Exception:
        wkey = None
    if wkey:
        open('wandb_api_key.txt', 'w').write(wkey.strip())
        print('wrote wandb key from colab secret')
    else:
        from google.colab import files
        files.upload()  # pick wandb_api_key.txt
print('have key file:', os.path.exists('wandb_api_key.txt'))


wrote wandb key from colab secret
have key file: True


## 5. Mount Google Drive

Outputs (results + checkpoints) are written straight to drive via `output_root`
in the config, so they survive a colab disconnect and a crashed run can resume.


In [5]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/inm363', exist_ok=True)
print('drive mounted -> /content/drive/MyDrive/inm363')


Mounted at /content/drive
drive mounted -> /content/drive/MyDrive/inm363


## 6. Build the data splits

Materialises gsm8k (sft), mmlu dev+val (grpo) and mmlu test (eval), and checks for leakage.


In [6]:
!python scripts/build_data.py


README.md: 100% 7.93k/7.93k [00:00<00:00, 1.89MB/s]
main/train-00000-of-00001.parquet: 100% 2.31M/2.31M [00:00<00:00, 2.83MB/s]
main/test-00000-of-00001.parquet: 100% 419k/419k [00:00<00:00, 563kB/s]
Generating train split: 100% 7473/7473 [00:00<00:00, 287309.54 examples/s]
Generating test split: 100% 1319/1319 [00:00<00:00, 251387.60 examples/s]
README.md: 100% 53.2k/53.2k [00:00<00:00, 93.3MB/s]
dataset_infos.json: 100% 138k/138k [00:00<00:00, 175MB/s]
high_school_psychology/test-00000-of-000(…): 100% 92.8k/92.8k [00:00<00:00, 168kB/s]
high_school_psychology/validation-00000-(…): 100% 15.2k/15.2k [00:00<00:00, 33.4kB/s]
high_school_psychology/dev-00000-of-0000(…): 100% 5.18k/5.18k [00:00<00:00, 9.94kB/s]
Generating test split: 100% 545/545 [00:00<00:00, 102062.58 examples/s]
Generating validation split: 100% 60/60 [00:00<00:00, 20745.05 examples/s]
Generating dev split: 100% 5/5 [00:00<00:00, 1976.58 examples/s]
marketing/test-00000-of-00001.parquet: 100% 37.3k/37.3k [00:00<00:00, 69

## 7. Wiring check

Quick sanity that the config loads, the gpu is seen, and the trackers resolve.


In [7]:
!python scripts/run.py --config configs/001_colab_full_b05.yaml


env=COLAB mode=FULL scale=B05 condition=SFT_GRPO
model=Qwen/Qwen2.5-0.5B device=cuda stages=['base_eval', 'sft', 'sft_eval', 'grpo', 'grpo_eval']
steps=500 N=8 seed=0
trackers=['tensorboard', 'wandb']
results -> /content/drive/MyDrive/inm363/results/COLAB_FULL_B05_sft_grpo_seed0/  checkpoints -> /content/drive/MyDrive/inm363/checkpoints/COLAB_FULL_B05_sft_grpo_seed0/
local smoke wiring ok


## 8. Full run

Base -> sft -> grpo -> eval, with metrics, per example preds, train logs, checkpoints,
manifest and wandb. Everything is written to drive as it goes.

**If it crashes or colab disconnects:** just re-run this cell. Finished stages are
skipped and a half-trained stage resumes from its last checkpoint on drive.

Watch grpo: frac_reward_zero_std should drop below 1, rewards/accuracy_reward/mean
should rise above 0, completions/clipped_ratio should stay low. The abort criteria were an early `clipped_ratio` above 0.3, flat reward with KL near zero by step 150, or a projected runtime over budget. The projection used the time per training step. If grpo OOMs on a T4, stop
the cell and drop the per device batch or the completion cap before rerunning,
or grab an L4 runtime.


In [ ]:
!python scripts/run.py --config configs/001_colab_full_b05.yaml --stage all


Streaming output truncated to the last 5000 lines.
│ │ best friend,  │ what was      │               │              │           │ │
│ │ who has never │ intended by   │               │              │           │ │
│ │ uttered this  │ the person's  │               │              │           │ │
│ │ sentiment     │ friend when   │               │              │           │ │
│ │ before. Your  │ he wrote the  │               │              │           │ │
│ │ friend's      │ following     │               │              │           │ │
│ │ comment best  │ sentence?     │               │              │           │ │
│ │ illustrates   │ What the      │               │              │           │ │
│ │ A.            │ subject's     │               │              │           │ │
│ │ overconfiden… │ psychology is │               │              │           │ │
│ │ B. a theory.  │ shown by what │               │              │           │ │
│ │ C. hindsight  │ the subject   │               │       

## 9. Peek at the results

Everything is saved under results/<tag>/ so I can re-analyse without rerunning.


In [ ]:
import json, os
from config import load_config, run_tag
from scripts.run import base_eval_dir
cfg = load_config('configs/001_colab_full_b05.yaml')
tag = run_tag(cfg)
base = os.path.join(cfg.output_root, 'results', tag)
bdir = base_eval_dir(cfg)
print('results dir:', base)
print('files:', os.listdir(base) if os.path.isdir(base) else 'missing')
print('shared base evals:', os.listdir(bdir) if os.path.isdir(bdir) else 'missing')
# only print what actually exists, the run might be mid way
for name in ('summary', 'manifest'):
    p = os.path.join(base, f'{name}.json')
    if os.path.exists(p):
        print(f'\n{name}:')
        print(json.dumps(json.load(open(p)), indent=2)[:2500])
    else:
        print(f'{name}.json not there yet (run not finished)')


results dir: /content/drive/MyDrive/inm363/results/COLAB_FULL_B05_sft_grpo_seed0
files: ['run.log', 'sft_trainlog.json', 'sft_preds.jsonl', 'sft_eval.json', 'sft_logprob_preds.jsonl', 'sft_logprob.json', 'grpo_trainlog.json', 'grpo_preds.jsonl', 'grpo_eval.json', 'grpo_logprob_preds.jsonl', 'grpo_logprob.json', 'summary.json', 'manifest.json']
shared base evals: ['base_eval.json', 'base_preds.jsonl', 'base_logprob_preds.jsonl', 'base_logprob.json']

summary:
{
  "scale": "B05",
  "mode": "FULL",
  "condition": "SFT_GRPO",
  "seed": 0,
  "base_acc": {
    "easy": 0.3117178612059158,
    "medium": 0.2939244663382594,
    "hard": 0.14646464646464646
  },
  "sft_acc": {
    "easy": 0.28896473265073946,
    "medium": 0.26436781609195403,
    "hard": 0.2222222222222222
  },
  "grpo_acc": {
    "easy": 0.27986348122866894,
    "medium": 0.22824302134646962,
    "hard": 0.22727272727272727
  },
  "reference": "sft",
  "grpo_delta": {
    "hard": 0.005050505050505055,
    "easy": -0.00910125142